In [1]:
import pandas as pd
import numpy as np

import sys
import os

# Get the current working directory (where the notebook is running)
notebook_dir = os.getcwd()

# Go up one level to the parent directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))

# Add parent directory to sys.path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Now you can import your module
from utils import factor_model_solution, factor_goodness_of_fit_test

*This notebook does factor analysis with k = 3 factors, trying to identify 2 pathways in which DICER1 and GART simultainuously.*
___
# $k = 2$ factors | GART & DICER1 pathways

## Loading and preparing data

See the `k1_[GENE]_initial_analysis.ipynb` notebooks for more intermediate steps than this notebook provides.

In [2]:
df_gene_effect = pd.read_csv("preprocessed_data/20Q4v2_Achilles_gene_effect.csv")
df_correlations = {
    "DICER1": pd.read_csv("preprocessed_data/corr_DICER1_preprocessed.csv", delimiter=";"),
    "GART": pd.read_csv("preprocessed_data/corr_GART_preprocessed.csv", delimiter=";")
}


In [3]:
# Number of genes to include from each corr file taking top X // 2 and bottom X // 2.
# Note that the same gene may be selected in both corr files. (But will only appear once in the final selected_genes list.)
number_of_genes = 40

assert number_of_genes % 2 == 0, "number_of_genes should be an even number."

selected_columns = pd.concat([df_correlations["DICER1"]["gene_evaluated"][:number_of_genes // 2],
                              df_correlations["DICER1"]["gene_evaluated"][-number_of_genes // 2:],
                              df_correlations["GART"]["gene_evaluated"][:number_of_genes // 2],
                              df_correlations["GART"]["gene_evaluated"][-number_of_genes // 2:]]).to_list()
selected_columns = set(selected_columns)

# number_of_genes multiplied by how many corr files we have.
number_of_duplicates = (number_of_genes * len(df_correlations)) - len(selected_columns) 
if number_of_duplicates > 0:
    print(f"WARNING: {number_of_duplicates} duplicates were removed.")

# Extract gene names according to the format of gene_effect dataset.
selected_columns_gene_effect_format = [column for column in df_gene_effect.columns if column.split(" ")[0] in selected_columns]

# Rough automatic check for whether we found all corresponding columns in gene_effect dataset that we've selected.
if not len(selected_columns) == len(selected_columns_gene_effect_format):
    print(f"WARNING: {len(selected_columns) - len(selected_columns_gene_effect_format)} column were not found in the database.")

# Extract columns from gene_effect dataset.
X = df_gene_effect[selected_columns_gene_effect_format].dropna() # Is dropping rows with na a good idea? Maybe use mean value instead?
X = X.to_numpy()
num_dropped_rows = df_gene_effect.shape[0] - X.shape[0]
if num_dropped_rows > 0:
    print(f"WARNING: {num_dropped_rows} rows contained at least 1 NA and was dropped.")
X.shape

(778, 78)

In [4]:
num_random_cols = 20
add_random = False

if add_random:
    X = np.hstack([X] + [np.random.random(size=X.shape[0]).reshape(-1, 1) for _ in range(num_random_cols)])

X.shape

(778, 78)

## Data Analysis

In [5]:
_, lambda_hat = factor_model_solution(X, k = 3)
lambda_hat

array([[-3.28441049e-02, -2.72060267e-03, -1.29947108e-01],
       [ 8.02280903e-01,  4.03375893e-04,  5.40349350e-02],
       [ 6.59504572e-02,  4.24338672e-01,  1.05499384e-01],
       [ 7.52102172e-02,  3.99285270e-01,  1.13307948e-01],
       [ 8.62020970e-01, -5.64322339e-03,  1.10565657e-01],
       [-6.08359298e-02,  1.72122936e-01, -3.94517447e-02],
       [-1.72669187e-03, -1.03124813e-01, -3.73584013e-02],
       [ 7.17384267e-01,  4.63073573e-02, -4.54514291e-01],
       [-3.73386470e-02,  5.43327365e-02,  1.60894557e-01],
       [-1.62023304e-02, -8.01053255e-02,  3.30314727e-02],
       [-1.45430620e-01,  2.99582199e-02,  1.44365278e-01],
       [-1.29699065e-01,  2.91516580e-01,  1.14036858e-01],
       [ 6.70644291e-01, -7.31464737e-02,  7.51597806e-02],
       [ 7.45635900e-01,  4.99391362e-02, -3.61222602e-01],
       [-1.00010153e-03,  8.00993439e-01,  9.31198908e-02],
       [-1.12669738e-01,  3.52532723e-01, -2.74447772e-02],
       [-5.40424285e-02,  2.70981515e-01

In [6]:
from utils import varimax

# loadings = lambda_hat.T # Extract loadings as a list

loadings = varimax(lambda_hat).T

In [7]:


# Extract gene names (everything before the space)
genes = [column.split(" ")[0] for column in selected_columns_gene_effect_format] 

if add_random:
    genes = genes + [str(i) + " random col" for i in range(num_random_cols)]

# Create a DataFrame from genes and values
df_values = pd.DataFrame({
    'gene_evaluated': genes,
    'loadings0': loadings[0],
    'loadings1': loadings[1],
    'loadings2': loadings[2]
})

# Merge with df_correlations on 'gene'
merged_df = (df_values
             .merge(df_correlations["DICER1"][['gene_evaluated', 'is_on_pathway']], on='gene_evaluated', how='left')
             .rename(columns={"is_on_pathway": "is_on_pathway_DICER1"})
             .merge(df_correlations["GART"][['gene_evaluated', 'is_on_pathway']], on='gene_evaluated', how='left')
             .rename(columns={"is_on_pathway": "is_on_pathway_GART"})
             )

In [8]:
# Allow pandas to display wider tables before linebreaking.
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

## Plotting

In [9]:
colors = []
for d, g in zip(merged_df['is_on_pathway_DICER1'], merged_df['is_on_pathway_GART']):
    if d == 1:
        colors.append("miRNA")
        continue

    if g == 1:
        colors.append("Purine Metabolism")
        continue

    colors.append("OTHER")

merged_df['category'] = colors

merged_df.sort_values(by='loadings0', ascending=False)[:]

,gene_evaluated,loadings0,loadings1,loadings2,is_on_pathway_DICER1,is_on_pathway_GART,category
65,SLC25A28,0.259910,-0.084002,-0.088490,0.0,0.0,OTHER
52,POP4,0.194720,0.108756,-0.130328,0.0,NaN,OTHER
24,HYOU1,0.166832,0.198273,-0.042148,0.0,0.0,OTHER
47,PCNT,0.139536,-0.050865,-0.029200,NaN,0.0,OTHER
28,MAP2K4,0.132321,-0.021737,-0.065952,NaN,0.0,OTHER
...,...,...,...,...,...,...,...
35,MTHFD1,-0.843013,0.031447,-0.041090,NaN,1.0,Purine Metabolism
4,ATIC,-0.869080,-0.005866,-0.001440,NaN,1.0,Purine Metabolism
21,GART,-0.877935,-0.049880,-0.095350,NaN,1.0,Purine Metabolism
53,PPAT,-0.878004,-0.097684,-0.145872,NaN,1.0,Purine Metabolism


In [13]:
import matplotlib.pyplot as plt

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

for label in ["miRNA", "Purine Metabolism", "OTHER"]:
    subset = merged_df[merged_df['category'] == label]
    ax.scatter(subset['loadings0'], subset['loadings1'], subset['loadings2'])

ax.view_init(azim=0)
plt.title("Factor Analysis on DICER1 pathway")
plt.legend()
plt.xlabel("1st Factor")
plt.show()

<IPython.core.display.Javascript object>

No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


In [ ]:
'''
READ ME:
The following code turns off

'''

# %matplotlib notebook
# from ipywidgets import *

# from matplotlib.animation import FuncAnimation
# import matplotlib.pyplot as plt
# import numpy as np

# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')

# ax.set_xlim(min(merged_df['loadings0']), max(merged_df['loadings0']))
# ax.set_ylim(min(merged_df['loadings1']), max(merged_df['loadings1']))
# ax.set_zlim(min(merged_df['loadings2']), max(merged_df['loadings2']))

# for label in ["miRNA", "Purine Metabolism", "OTHER"]:
#     subset = merged_df[merged_df['category'] == label]
#     ax.scatter(subset['loadings0'], subset['loadings1'], subset['loadings2'], label=label)
# # line = ax.scatter(merged_df['loadings0'], merged_df['loadings1'], merged_df['loadings2'])
# plt.legend(loc='upper right')
# plt.title("Combined Pathways - miRNA & Purine Metabolism")
# def animation(i):
#     ax.view_init(azim=i)
#     return ax,

# ani=FuncAnimation(fig, animation, frames=np.arange(0, 360),interval=15)
# plt.show()

# interact()
# plt.show()
# ani.save("test.gif")

<IPython.core.display.Javascript object>

MovieWriter ffmpeg unavailable; using Pillow instead.
